# 04 — Interactive PIC-Flow inference

Live sliders for device geometry + Euler step count. The model re-runs on every slider release; predicted $|E_z|$ updates inline.

**Tip:** start with `Euler steps = 20` for snappy real-time feedback (≈ 0.4 s per inference on an A100). Crank to 100+ once you've found a geometry you like, for paper-grade quality.

Geometry rasterization (Meep) is cached per (device, params, source port) tuple — sliding back to a previous combo is instant.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = Path('..').resolve()
for p in (REPO, REPO / 'Model', REPO / 'tools', REPO / 'FDTD'):
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

from huggingface_hub import hf_hub_download, try_to_load_from_cache
from predict_parametric_device import (
    DEFAULT_CROP_X_PX, DEFAULT_CROP_Y_PX, DEFAULT_DPML, DEFAULT_RESOLUTION, DEFAULT_WAVELENGTH_UM,
    _build_cond_maps, _build_cond_vector, _build_model_from_checkpoint, _checkpoint_state_dict,
    _make_device_and_arrays,
)
from flow_matching import sample as fm_sample
from unified_sweep import PARAM_RANGES, INPUT_PORTS

HF_REPO = 'RizzoLab/PIC-Flow'
HF_FILE = 'checkpoints/phase_residual_300.pt'

ckpt_path = try_to_load_from_cache(repo_id=HF_REPO, filename=HF_FILE)
if ckpt_path is None or ckpt_path is False:
    print('Downloading checkpoint from Hugging Face (~1 GB, one-time)...')
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, repo_type='model')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
model = _build_model_from_checkpoint(ckpt, device=device)
_, state = _checkpoint_state_dict(ckpt, use_ema=True)
model.load_state_dict(state, strict=True)
model.eval()
stats = ckpt['stats']
ckpt_args = ckpt.get('args')
time_grid = str(getattr(ckpt_args, 'time_grid', 'linear')) if ckpt_args is not None else 'linear'
print(f'model loaded on {device} ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)')

In [ ]:
_GEOM_CACHE = {}

def rasterize(device_type, params_tuple, source_port, wavelength_um=DEFAULT_WAVELENGTH_UM):
    key = (device_type, params_tuple, int(source_port), float(wavelength_um))
    if key in _GEOM_CACHE:
        return _GEOM_CACHE[key]
    _, eps, src, _, _, cell = _make_device_and_arrays(
        device_type=device_type, params=dict(params_tuple),
        source_port=int(source_port), wavelength_um=float(wavelength_um),
        resolution=DEFAULT_RESOLUTION, dpml=DEFAULT_DPML,
        crop_x_px=DEFAULT_CROP_X_PX, crop_y_px=DEFAULT_CROP_Y_PX,
    )
    eps = np.asarray(eps, dtype=np.float32)
    src = np.asarray(src, dtype=np.float32)
    _GEOM_CACHE[key] = (eps, src, cell)
    return _GEOM_CACHE[key]


def infer_field(eps, src_mask, cell_size, num_steps, seed=0,
                wavelength_um=DEFAULT_WAVELENGTH_UM):
    dx_um = cell_size[0] / eps.shape[1]
    dy_um = cell_size[1] / eps.shape[0]
    cond_maps = _build_cond_maps(
        eps, src_mask, stats=stats, ckpt_args=ckpt_args,
        device=device, dx_um=dx_um, dy_um=dy_um,
    )
    cond_v = _build_cond_vector(wavelength_um, stats, device=device)
    lam = torch.tensor([[wavelength_um]], device=device, dtype=torch.float32)

    gen = torch.Generator(device=device); gen.manual_seed(int(seed))
    x0 = torch.randn((1, 2, eps.shape[0], eps.shape[1]),
                     device=device, dtype=torch.float32, generator=gen)
    use_amp = (device.type == 'cuda')
    with torch.no_grad():
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            x1 = fm_sample(model, x0, num_steps=int(num_steps),
                           cond_maps=cond_maps, cond=cond_v, lambda_um=lam,
                           time_grid=time_grid)
    fn = x1.float()[0].cpu().numpy()
    ezr = fn[0] * float(stats['ez_real_std']) + float(stats['ez_real_mean'])
    ezi = fn[1] * float(stats['ez_imag_std']) + float(stats['ez_imag_mean'])
    return np.abs(ezr + 1j * ezi).astype(np.float32)

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output

DEVICES = ('mmi', 'ybranch', 'directional_coupler')
DEVICE_LABELS = {'mmi': '2x2 MMI', 'ybranch': 'Y-branch', 'directional_coupler': 'Directional coupler'}
PARAM_QUANTUM = 0.025  # um, matches the dataset's Latin-hypercube quantization
SLIDER_LAYOUT = W.Layout(width='460px')
SLIDER_STYLE = {'description_width': '180px'}

device_dd = W.Dropdown(options=[(DEVICE_LABELS[d], d) for d in DEVICES],
                       value='directional_coupler', description='Device')
port_dd = W.Dropdown(options=INPUT_PORTS['directional_coupler'], value=1, description='Source port')
steps_w = W.IntSlider(min=1, max=200, value=20, step=1, description='Euler steps',
                       continuous_update=False, layout=SLIDER_LAYOUT, style=SLIDER_STYLE)
seed_w = W.IntText(value=0, description='Seed', layout=W.Layout(width='180px'))
auto_w = W.Checkbox(value=True, description='Auto-update', indent=False,
                     layout=W.Layout(width='130px'))
run_btn = W.Button(description='Run inference', button_style='primary', icon='play',
                    layout=W.Layout(width='160px'))

slider_box = W.VBox([])
sliders = {}
out = W.Output()

_busy = {'flag': False}  # avoid re-entrant render storms while a render is mid-flight

def make_sliders(device_type):
    return {
        name: W.FloatSlider(
            min=float(lo), max=float(hi),
            value=round(((lo + hi) * 0.5) / PARAM_QUANTUM) * PARAM_QUANTUM,
            step=PARAM_QUANTUM, description=name, readout_format='.3f',
            continuous_update=False, layout=SLIDER_LAYOUT, style=SLIDER_STYLE,
        )
        for name, (lo, hi) in PARAM_RANGES[device_type].items()
    }

def render(*_):
    if _busy['flag']:
        return
    _busy['flag'] = True
    try:
        with out:
            clear_output(wait=True)
            params_t = tuple(sorted((n, float(s.value)) for n, s in sliders.items()))
            t0 = time.perf_counter()
            eps, src, cell = rasterize(device_dd.value, params_t, port_dd.value)
            t_geom = time.perf_counter() - t0
            t1 = time.perf_counter()
            mag = infer_field(eps, src, cell, num_steps=steps_w.value, seed=seed_w.value)
            t_inf = time.perf_counter() - t1

            fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.0), constrained_layout=True)
            ax = axes[0]
            ax.imshow(eps, origin='lower', cmap='viridis', interpolation='nearest', aspect='equal')
            ax.imshow(np.ma.masked_where(src <= 0.5, np.ones_like(src, dtype=np.float32)),
                      origin='lower', cmap='Greens', vmin=0, vmax=1, alpha=0.55,
                      interpolation='nearest', aspect='equal')
            ax.set_title(f"Geometry $\\varepsilon_r$  (raster {t_geom*1000:.0f} ms{' cached' if t_geom < 0.05 else ''})")
            ax.set_xticks([]); ax.set_yticks([])
            ax = axes[1]
            vmax = float(np.percentile(mag, 99.5))
            ax.imshow(mag, origin='lower', cmap='magma', vmin=0, vmax=max(vmax, 1e-6),
                      interpolation='nearest', aspect='equal')
            ax.set_title(f"PIC-Flow $|E_z|$  ({steps_w.value} Euler steps, {t_inf*1000:.0f} ms)")
            ax.set_xticks([]); ax.set_yticks([])
            plt.show()
    finally:
        _busy['flag'] = False

def maybe_render(*_):
    if auto_w.value:
        render()

def attach_param_observers():
    for s in sliders.values():
        s.observe(maybe_render, names='value')

def rebuild_for_device(*_):
    global sliders
    sliders = make_sliders(device_dd.value)
    attach_param_observers()
    port_dd.unobserve(maybe_render, names='value')
    port_dd.options = INPUT_PORTS[device_dd.value]
    port_dd.value = port_dd.options[0]
    port_dd.observe(maybe_render, names='value')
    slider_box.children = tuple(sliders.values())
    if auto_w.value:
        render()

device_dd.observe(rebuild_for_device, names='value')
port_dd.observe(maybe_render, names='value')
steps_w.observe(maybe_render, names='value')
seed_w.observe(maybe_render, names='value')
run_btn.on_click(lambda *_: render())

rebuild_for_device()

display(W.VBox([
    W.HBox([device_dd, port_dd, auto_w, run_btn]),
    slider_box,
    W.HBox([steps_w, seed_w]),
    out,
]))

## Notes on speed and quality

| Euler steps | A100 wall time | Compliance $\rho_R$ on test set | Use case |
|---|---|---|---|
| 1 | 22 ms | 14.5% | one-step preview, very rough |
| 5 | 110 ms | 5.5% | snappy slider feedback |
| 20 | 440 ms | 3.0% | recommended default for live exploration |
| 50 | 1.1 s | 2.1% | high quality |
| 100 | 2.2 s | 1.9% | paper-grade |
| 200 | 4.4 s | 1.2% | matches the headline FM+phase+residual numbers |

Compliance numbers from the in-distribution wall-clock benchmark in the paper (Table V).

**Geometry caching:** moving a slider back to a previously visited combination is free — the rasterized $\varepsilon$ and source-mask are cached in `_GEOM_CACHE`. Clear with `_GEOM_CACHE.clear()`.

**Auto-update off:** uncheck *Auto-update* if you want to set several sliders before triggering inference; click **Run inference** to render on demand.